In [1]:
!uv pip install ultralytics
from ultralytics import YOLO

Using Python 3.13.15 environment at: /usr
Checked 1 package in 139ms


In [5]:
import urllib.request
import zipfile
import os

url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip'
output_filename = 'construction-ppe.zip'
extract_dir = 'construction-ppe'

print("Downloading dataset...")
urllib.request.urlretrieve(url, output_filename)
print(f"Downloaded to {output_filename}")

print("Extracting dataset...")
with zipfile.ZipFile(output_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print(f"Extracted successfully to ./{extract_dir}")

Downloaded to construction-ppe.zip
Extracting dataset...
Extracted successfully to ./construction-ppe


In [3]:
#GENERATING THE MODEL (YOLO 26n)
model = YOLO("yolo26n.pt")
results = model.train(data="/content/construction-ppe/data.yaml", epochs=100, imgsz=640,scale=0.5, classes= [0,1,2,4,6],
    project="/content/drive/MyDrive/yolo_runs", name="ppe")   # saves to Drive)

Ultralytics 8.4.180 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=[0, 1, 2, 4, 6], close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/construction-ppe/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe-4, nbs=64, nms=None, opset=None, optim

In [4]:
from ultralytics import YOLO

# Load a model
model_PPE = YOLO(f"{model.trainer.save_dir}/weights/best.pt")  # load a fine-tuned model

# Export the model
model_PPE.export(format="onnx")

Ultralytics 8.4.180 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 121 layers, 2,392,165 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/yolo_runs/ppe-4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 15, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 231ms
Prepared 4 packages in 520ms
Installed 4 packages in 33ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.31.0
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX:

'/content/drive/MyDrive/yolo_runs/ppe-4/weights/best.onnx'

In [6]:
import os
import glob
import shutil
from google.colab import files

# Path where YOLO runs are saved on Google Drive
base_dir = "/content/drive/MyDrive/yolo_runs"

# Find all matching 'ppe' directories
run_dirs = glob.glob(os.path.join(base_dir, "ppe*"))

if not run_dirs:
    print("No PPE training directories found in Google Drive.")
else:
    # Sort by creation/modification time to get the latest folder (e.g., ppe-4)
    latest_dir = max(run_dirs, key=os.path.getmtime)
    folder_name = os.path.basename(latest_dir)
    print(f"Latest PPE run folder detected: {latest_dir}")

    # Archive name
    archive_name = f"/content/{folder_name}_results"

    print(f"Creating zip archive of {latest_dir}...")
    # shutil.make_archive creates a fast zip file directly from Google Drive path to local disk
    shutil.make_archive(archive_name, 'zip', latest_dir)
    zip_file_path = f"{archive_name}.zip"
    print(f"Created archive at: {zip_file_path}")

    # Fast browser download trigger
    print("Triggering fast browser download of the zip file...")
    files.download(zip_file_path)

Latest PPE run folder detected: /content/drive/MyDrive/yolo_runs/ppe-4
Creating zip archive of /content/drive/MyDrive/yolo_runs/ppe-4...
Created archive at: /content/ppe-4_results.zip
Triggering fast browser download of the zip file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from ultralytics import YOLO
import numpy as np
import cv2
import io
from PIL import Image as PILImage

# 1. Initialize the YOLO model with your latest trained weights
model = YOLO('/content/drive/MyDrive/yolo_runs/ppe/weights/best.pt')

# 2. JavaScript helper to access the webcam and stream images to Python
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelVal = "";

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
    }

    function createDom() {
       div = document.createElement('div');
       div.style.border = '2px solid black';
       div.style.padding = '10px';
       div.style.width = '640px';
       div.style.borderRadius = '5px';

       video = document.createElement('video');
       video.style.display = 'block';
       video.width = 640;
       video.height = 480;
       video.autoplay = true;
       video.playsInline = true;

       var stopBtn = document.createElement('button');
       stopBtn.textContent = "Stop Webcam";
       stopBtn.style.marginTop = '10px';
       stopBtn.style.padding = '5px 10px';
       stopBtn.style.cursor = 'pointer';
       stopBtn.onclick = function() {
          removeDom();
       };

       imgElement = document.createElement('img');
       imgElement.style.display = 'block';
       imgElement.width = 640;
       imgElement.height = 480;

       div.appendChild(video);
       div.appendChild(stopBtn);
       div.appendChild(imgElement);
       document.body.appendChild(div);
    }

    async function initWebcam() {
       createDom();
       stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});
       video.srcObject = stream;
       await video.play();

       captureCanvas = document.createElement('canvas');
       captureCanvas.width = 640;
       captureCanvas.height = 480;
    }

    async function captureFrame() {
       if (!video) return null;
       var ctx = captureCanvas.getContext('2d');
       ctx.drawImage(video, 0, 0, 640, 480);
       return captureCanvas.toDataURL('image/jpeg', 0.8);
     }

     function updateImage(src) {
       if (imgElement) {
          imgElement.src = src;
       }
     }
    ''')
  display(js)

def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

def bbox_to_bytes(bbox_array):
  _, encoded_img = cv2.imencode('.jpg', bbox_array)
  b64_string = 'data:image/jpeg;base64,' + b64encode(encoded_img).decode('utf-8')
  return b64_string

# 3. Main real-time prediction loop
def run_live_detection():
  # Start the browser-side webcam setup
  eval_js('initWebcam()')

  while True:
    # Grab a frame from the webcam via JS
    js_reply = eval_js('captureFrame()')
    if not js_reply:
      break

    # Convert JS image to OpenCV format
    frame = js_to_image(js_reply)

    # Run YOLOv8 detection on the frame with confidence threshold set to 0.5
    results = model(frame, conf=0.3, verbose=False)

    # Plot predictions directly onto the frame
    annotated_frame = results[0].plot()

    # Send the frame with bounding boxes back to the browser interface
    bbox_bytes = bbox_to_bytes(annotated_frame)
    eval_js(f'updateImage("{bbox_bytes}")')

# Run the stream
video_stream()
run_live_detection()

<IPython.core.display.Javascript object>